In [1]:
import os
import glob
import numpy as np
import scipy.io as sio

# -----------------------------
# Paths
# -----------------------------
SRC_ROOT = r"G:\Data Sets\Cutting tool organized\Vibration_1320_1"   # Source (use 100% for training)
TGT_ROOT = r"G:\Data Sets\Cutting tool organized\Vibration_1440_1"   # Target (80% train, 20% test)

class_names = ["BF", "GF", "N", "TF"]

def list_mat_files(root_dir):
    all_paths = {}
    for cname in class_names:
        pattern = os.path.join(root_dir, f"Vib_{cname}*", "*.mat")
        files = sorted(glob.glob(pattern))
        all_paths[cname] = files
    return all_paths

def print_summary(tag, root_dir):
    print("\n==============================")
    print(f"{tag} ROOT: {root_dir}")
    print("==============================")
    paths = list_mat_files(root_dir)
    total = 0
    for cname in class_names:
        n = len(paths[cname])
        total += n
        print(f"{cname}: {n} files")
    print(f"TOTAL: {total} files")
    return paths

# 1) Count files
src_paths = print_summary("SOURCE", SRC_ROOT)
tgt_paths = print_summary("TARGET", TGT_ROOT)

# 2) Try loading ONE example file from each domain
def try_load_one(paths_dict, tag):
    # pick first available file among classes
    example = None
    for cname in class_names:
        if len(paths_dict[cname]) > 0:
            example = paths_dict[cname][0]
            break

    if example is None:
        print(f"\n[{tag}] No .mat files found to test loading.")
        return

    print(f"\n[{tag}] Loading example file:\n{example}")
    mat = sio.loadmat(example)

    # print keys (remove MATLAB meta keys)
    keys = [k for k in mat.keys() if not k.startswith("__")]
    print(f"[{tag}] Keys found:", keys)

    if "signals" in mat:
        sig = np.asarray(mat["signals"])
        print(f"[{tag}] signals shape: {sig.shape}, dtype: {sig.dtype}")
    else:
        print(f"[{tag}] 'signals' not found!")

    if "fs" in mat:
        fs = np.asarray(mat["fs"])
        print(f"[{tag}] fs shape: {fs.shape}, value: {fs.squeeze()}")
    else:
        print(f"[{tag}] 'fs' not found!")

try_load_one(src_paths, "SOURCE")
try_load_one(tgt_paths, "TARGET")



SOURCE ROOT: G:\Data Sets\Cutting tool organized\Vibration_1320_1
BF: 112 files
GF: 111 files
N: 116 files
TF: 112 files
TOTAL: 451 files

TARGET ROOT: G:\Data Sets\Cutting tool organized\Vibration_1440_1
BF: 112 files
GF: 110 files
N: 112 files
TF: 112 files
TOTAL: 446 files

[SOURCE] Loading example file:
G:\Data Sets\Cutting tool organized\Vibration_1320_1\Vib_BF1320_1\20250212_112959_Vib.mat
[SOURCE] Keys found: ['signals', 'fs']
[SOURCE] signals shape: (2, 25600), dtype: float64
[SOURCE] fs shape: (1, 1), value: 25600.0

[TARGET] Loading example file:
G:\Data Sets\Cutting tool organized\Vibration_1440_1\Vib_BF1440_1\20250212_133259_Vib.mat
[TARGET] Keys found: ['signals', 'fs']
[TARGET] signals shape: (2, 25600), dtype: float64
[TARGET] fs shape: (1, 1), value: 25600.0


In [2]:
import os
import glob
import numpy as np

# -----------------------------
# Paths
# -----------------------------
TGT_ROOT = r"G:\Data Sets\Cutting tool organized\Vibration_1440_1"
SAVE_DIR = r"F:\MCT PAPER_FAROOQ"
SPLIT_DIR = os.path.join(SAVE_DIR, "splits")
os.makedirs(SPLIT_DIR, exist_ok=True)

class_names = ["BF", "GF", "N", "TF"]
SEED = 42
rng = np.random.RandomState(SEED)

def list_mat_files(root_dir):
    all_paths = {}
    for cname in class_names:
        pattern = os.path.join(root_dir, f"Vib_{cname}*", "*.mat")
        files = sorted(glob.glob(pattern))
        all_paths[cname] = files
    return all_paths

tgt_paths = list_mat_files(TGT_ROOT)

train_files = []
test_files = []

print("\n==============================")
print("TARGET 80/20 SPLIT (file-level)")
print("==============================")

for cname in class_names:
    files = tgt_paths[cname]
    n = len(files)
    if n == 0:
        raise RuntimeError(f"No files found for class {cname} in {TGT_ROOT}")

    idx = np.arange(n)
    rng.shuffle(idx)

    n_train = int(np.floor(0.8 * n))
    tr = [files[i] for i in idx[:n_train]]
    te = [files[i] for i in idx[n_train:]]

    train_files.extend(tr)
    test_files.extend(te)

    print(f"{cname}: total={n:3d} | train={len(tr):3d} | test={len(te):3d}")

print("\nTOTAL train files:", len(train_files))
print("TOTAL test files :", len(test_files))

# Save split lists
train_list_path = os.path.join(SPLIT_DIR, "target_train_files.txt")
test_list_path  = os.path.join(SPLIT_DIR, "target_test_files.txt")

with open(train_list_path, "w", encoding="utf-8") as f:
    for p in train_files:
        f.write(p + "\n")

with open(test_list_path, "w", encoding="utf-8") as f:
    for p in test_files:
        f.write(p + "\n")

print("\nSaved:")
print(" ", train_list_path)
print(" ", test_list_path)



TARGET 80/20 SPLIT (file-level)
BF: total=112 | train= 89 | test= 23
GF: total=110 | train= 88 | test= 22
N: total=112 | train= 89 | test= 23
TF: total=112 | train= 89 | test= 23

TOTAL train files: 355
TOTAL test files : 91

Saved:
  F:\MCT PAPER_FAROOQ\splits\target_train_files.txt
  F:\MCT PAPER_FAROOQ\splits\target_test_files.txt


In [3]:
import os
import numpy as np
import scipy.io as sio

# -----------------------------
# Paths
# -----------------------------
SRC_ROOT = r"G:\Data Sets\Cutting tool organized\Vibration_1320_1"
TGT_ROOT = r"G:\Data Sets\Cutting tool organized\Vibration_1440_1"
SAVE_DIR = r"F:\MCT PAPER_FAROOQ"

train_list_path = os.path.join(SAVE_DIR, "splits", "target_train_files.txt")
test_list_path  = os.path.join(SAVE_DIR, "splits", "target_test_files.txt")

class_names = ["BF", "GF", "N", "TF"]
name_to_label = {c:i for i,c in enumerate(class_names)}

# segmentation
WIN  = 2048
STEP = 100

def load_one_file(filepath, sensor_idx=0):
    mat = sio.loadmat(filepath)
    sig = np.asarray(mat["signals"]).astype(np.float32)  # (2, 25600)
    return sig[sensor_idx, :]

def segment_signal(x, win=2048, step=100):
    N = len(x)
    if N < win:
        return np.empty((0, win), dtype=np.float32)
    idxs = np.arange(0, N - win + 1, step)
    segs = np.stack([x[i:i+win] for i in idxs], axis=0).astype(np.float32)
    return segs

def infer_label_from_path(filepath):
    # expects folder name containing Vib_BF..., Vib_GF..., Vib_N..., Vib_TF...
    base = os.path.basename(os.path.dirname(filepath))  # class folder
    # examples: Vib_BF1320_1, Vib_GF1440_1, Vib_N1320_1, Vib_TF1440_1
    for cname in class_names:
        if f"Vib_{cname}" in base:
            return name_to_label[cname]
    raise ValueError(f"Cannot infer class label from folder name: {base}")

def read_file_list(txt_path):
    with open(txt_path, "r", encoding="utf-8") as f:
        files = [line.strip() for line in f.readlines() if line.strip()]
    return files

def build_from_file_list(files, sensor_idx=0):
    X_list, y_list = [], []
    per_class_counts = {c:0 for c in class_names}
    total_files_used = 0

    for fp in files:
        y = infer_label_from_path(fp)
        x = load_one_file(fp, sensor_idx=sensor_idx)
        segs = segment_signal(x, win=WIN, step=STEP)
        if segs.shape[0] == 0:
            continue

        X_list.append(segs)
        y_list.append(np.full((segs.shape[0],), y, dtype=np.int64))

        cname = class_names[y]
        per_class_counts[cname] += segs.shape[0]
        total_files_used += 1

    X = np.concatenate(X_list, axis=0) if len(X_list) else np.empty((0, WIN), dtype=np.float32)
    y = np.concatenate(y_list, axis=0) if len(y_list) else np.empty((0,), dtype=np.int64)
    return X, y, per_class_counts, total_files_used

# -----------------------------
# Source: use ALL files for training
# -----------------------------
# We rebuild file list from the folder, not from txt, since you want 100% source
import glob
def list_source_all_files(src_root):
    all_files = []
    for cname in class_names:
        pattern = os.path.join(src_root, f"Vib_{cname}*", "*.mat")
        all_files.extend(sorted(glob.glob(pattern)))
    return all_files

src_files_all = list_source_all_files(SRC_ROOT)
tgt_train_files = read_file_list(train_list_path)
tgt_test_files  = read_file_list(test_list_path)

print("SOURCE files:", len(src_files_all))
print("TARGET train files:", len(tgt_train_files))
print("TARGET test files :", len(tgt_test_files))

# Build segments
Xs, ys, src_counts, src_used = build_from_file_list(src_files_all, sensor_idx=0)
Xt_tr, yt_tr, tgttr_counts, tgttr_used = build_from_file_list(tgt_train_files, sensor_idx=0)
Xt_te, yt_te, tgtte_counts, tgtte_used = build_from_file_list(tgt_test_files, sensor_idx=0)

print("\n==============================")
print("SEGMENT SUMMARY (WIN=2048, STEP=100, sensor=0)")
print("==============================")

print(f"SOURCE train segments: {Xs.shape[0]}  | files used: {src_used}")
print(" per-class segments:", src_counts)

print(f"\nTARGET train segments (unlabeled in training): {Xt_tr.shape[0]} | files used: {tgttr_used}")
print(" per-class segments:", tgttr_counts)

print(f"\nTARGET test segments: {Xt_te.shape[0]} | files used: {tgtte_used}")
print(" per-class segments:", tgtte_counts)

# Save labels for later evaluation consistency
np.save(os.path.join(SAVE_DIR, "ys_source.npy"), ys)
np.save(os.path.join(SAVE_DIR, "yt_target_train.npy"), yt_tr)
np.save(os.path.join(SAVE_DIR, "yt_target_test.npy"), yt_te)

print("\nSaved label arrays to:", SAVE_DIR)
print("  ys_source.npy")
print("  yt_target_train.npy")
print("  yt_target_test.npy")


SOURCE files: 451
TARGET train files: 355
TARGET test files : 91

SEGMENT SUMMARY (WIN=2048, STEP=100, sensor=0)
SOURCE train segments: 106436  | files used: 451
 per-class segments: {'BF': 26432, 'GF': 26196, 'N': 27376, 'TF': 26432}

TARGET train segments (unlabeled in training): 83780 | files used: 355
 per-class segments: {'BF': 21004, 'GF': 20768, 'N': 21004, 'TF': 21004}

TARGET test segments: 21476 | files used: 91
 per-class segments: {'BF': 5428, 'GF': 5192, 'N': 5428, 'TF': 5428}

Saved label arrays to: F:\MCT PAPER_FAROOQ
  ys_source.npy
  yt_target_train.npy
  yt_target_test.npy


In [4]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

# -----------------------------
# Load the arrays created in Step 3
# -----------------------------
# IMPORTANT:
# In your notebook, these variables Xs, ys, Xt_tr, yt_tr, Xt_te, yt_te
# already exist from Step 3. If you restarted the kernel, you must reload them.
#
# If you restarted, re-run Step 3 or save/load X arrays too.
# Here I assume Step 3 variables are still in memory.

# -----------------------------
# Config
# -----------------------------
BATCH = 64
NUM_WORKERS = 0  # keep 0 on Windows to avoid issues
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -----------------------------
# Dataset
# -----------------------------
class VibSegDataset(Dataset):
    def __init__(self, X, y=None):
        # X: (N, WIN)
        self.X = X.astype(np.float32)
        self.y = None if y is None else y.astype(np.int64)

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        # return shape as (1, WIN) for 1D CNN input
        x = self.X[idx][None, :]  # (1, WIN)
        x = torch.tensor(x, dtype=torch.float32)
        if self.y is None:
            return x
        y = torch.tensor(self.y[idx], dtype=torch.long)
        return x, y

# -----------------------------
# Build loaders
# -----------------------------
src_ds = VibSegDataset(Xs, ys)          # labeled
tgt_tr_ds = VibSegDataset(Xt_tr, None)  # unlabeled in training
tgt_te_ds = VibSegDataset(Xt_te, yt_te) # labeled for evaluation

src_loader = DataLoader(src_ds, batch_size=BATCH, shuffle=True, num_workers=NUM_WORKERS, drop_last=True)
tgt_tr_loader = DataLoader(tgt_tr_ds, batch_size=BATCH, shuffle=True, num_workers=NUM_WORKERS, drop_last=True)
tgt_te_loader = DataLoader(tgt_te_ds, batch_size=BATCH, shuffle=False, num_workers=NUM_WORKERS, drop_last=False)

print("Device:", device)
print("Source loader batches:", len(src_loader))
print("Target train loader batches:", len(tgt_tr_loader))
print("Target test loader batches:", len(tgt_te_loader))

# -----------------------------
# Sanity check one batch
# -----------------------------
xs, ys_b = next(iter(src_loader))
xt = next(iter(tgt_tr_loader))
xte, yte = next(iter(tgt_te_loader))

print("\nBatch shapes check:")
print(" Source batch X:", xs.shape, "y:", ys_b.shape)
print(" Target train batch X:", xt.shape)
print(" Target test batch X:", xte.shape, "y:", yte.shape)

# move one batch to device to confirm
xs = xs.to(device)
print("\nMoved source batch to device:", xs.device)


Device: cpu
Source loader batches: 1663
Target train loader batches: 1309
Target test loader batches: 336

Batch shapes check:
 Source batch X: torch.Size([64, 1, 2048]) y: torch.Size([64])
 Target train batch X: torch.Size([64, 1, 2048])
 Target test batch X: torch.Size([64, 1, 2048]) y: torch.Size([64])

Moved source batch to device: cpu


In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# -----------------------------
# SKN block for 1D signals
# (selective kernel: two branches with different kernel sizes)
# -----------------------------
class SKN1D(nn.Module):
    def __init__(self, channels, reduction=16, k1=3, k2=7):
        super().__init__()
        pad1 = k1 // 2
        pad2 = k2 // 2

        self.branch1 = nn.Sequential(
            nn.Conv1d(channels, channels, kernel_size=k1, padding=pad1, bias=False),
            nn.BatchNorm1d(channels),
            nn.ReLU(inplace=True)
        )
        self.branch2 = nn.Sequential(
            nn.Conv1d(channels, channels, kernel_size=k2, padding=pad2, bias=False),
            nn.BatchNorm1d(channels),
            nn.ReLU(inplace=True)
        )

        mid = max(channels // reduction, 8)
        self.fc = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),                 # (B,C,1)
            nn.Conv1d(channels, mid, kernel_size=1), # (B,mid,1)
            nn.ReLU(inplace=True)
        )
        self.attn1 = nn.Conv1d(mid, channels, kernel_size=1)
        self.attn2 = nn.Conv1d(mid, channels, kernel_size=1)

    def forward(self, x):
        x1 = self.branch1(x)
        x2 = self.branch2(x)

        u = x1 + x2
        z = self.fc(u)

        a1 = self.attn1(z)
        a2 = self.attn2(z)

        a = torch.stack([a1, a2], dim=1)     # (B,2,C,1)
        a = F.softmax(a, dim=1)

        out = a[:, 0] * x1 + a[:, 1] * x2    # (B,C,L)
        return out

# -----------------------------
# CNN backbone + SKN + classifier
# -----------------------------
class SKN_CNN_Classifier(nn.Module):
    def __init__(self, n_classes=4, feat_dim=128):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=11, stride=2, padding=5, bias=False),
            nn.BatchNorm1d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(kernel_size=2)
        )

        self.block1 = nn.Sequential(
            nn.Conv1d(32, 64, kernel_size=7, padding=3, bias=False),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(kernel_size=2),
            SKN1D(64, reduction=16, k1=3, k2=7)
        )

        self.block2 = nn.Sequential(
            nn.Conv1d(64, 128, kernel_size=5, padding=2, bias=False),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(kernel_size=2),
            SKN1D(128, reduction=16, k1=3, k2=7)
        )

        self.gap = nn.AdaptiveAvgPool1d(1)
        self.feat = nn.Linear(128, feat_dim)
        self.cls  = nn.Linear(feat_dim, n_classes)

    def forward(self, x):
        # x: (B,1,2048)
        x = self.stem(x)
        x = self.block1(x)
        x = self.block2(x)
        x = self.gap(x).squeeze(-1)          # (B,128)
        f = self.feat(x)                     # (B,feat_dim)
        f = F.relu(f, inplace=True)
        logits = self.cls(f)                 # (B,n_classes)
        return f, logits

# -----------------------------
# Sanity forward pass
# -----------------------------
model = SKN_CNN_Classifier(n_classes=4, feat_dim=128).to(device)
model.eval()

# use the loaders you built in Step 4
xs, ys_b = next(iter(src_loader))
xt = next(iter(tgt_tr_loader))

xs = xs.to(device)
xt = xt.to(device)

with torch.no_grad():
    fs, logits_s = model(xs)
    ft, logits_t = model(xt)

print("\nForward pass shapes:")
print(" Source features:", fs.shape, "Source logits:", logits_s.shape)
print(" Target features:", ft.shape, "Target logits:", logits_t.shape)


Device: cpu

Forward pass shapes:
 Source features: torch.Size([64, 128]) Source logits: torch.Size([64, 4])
 Target features: torch.Size([64, 128]) Target logits: torch.Size([64, 4])


In [6]:
import torch
import torch.nn.functional as F

# -----------------------------
# RBF kernel
# -----------------------------
def gaussian_kernel(source, target, kernel_mul=2.0, kernel_num=5, fix_sigma=None):
    """
    source: (bs, d)
    target: (bt, d)
    return: (bs+bt, bs+bt)
    """
    n_samples = source.size(0) + target.size(0)
    total = torch.cat([source, target], dim=0)  # (n, d)

    total0 = total.unsqueeze(0).expand(total.size(0), total.size(0), total.size(1))
    total1 = total.unsqueeze(1).expand(total.size(0), total.size(0), total.size(1))
    L2_distance = ((total0 - total1) ** 2).sum(2)  # (n, n)

    if fix_sigma:
        bandwidth = fix_sigma
    else:
        bandwidth = torch.sum(L2_distance.detach()) / (n_samples**2 - n_samples + 1e-12)

    bandwidth = bandwidth / (kernel_mul ** (kernel_num // 2))
    bandwidth_list = [bandwidth * (kernel_mul ** i) for i in range(kernel_num)]

    kernel_val = [torch.exp(-L2_distance / (bw + 1e-12)) for bw in bandwidth_list]
    return sum(kernel_val)  # (n, n)

# -----------------------------
# LMMD loss
# -----------------------------
def lmmd_loss(source_feat, target_feat, source_label, target_prob, num_classes=4):
    """
    source_feat: (bs, d)
    target_feat: (bt, d)
    source_label: (bs,)
    target_prob: (bt, C) softmax probabilities
    """
    bs = source_feat.size(0)
    bt = target_feat.size(0)

    K = gaussian_kernel(source_feat, target_feat)  # (bs+bt, bs+bt)
    K_ss = K[:bs, :bs]
    K_tt = K[bs:, bs:]
    K_st = K[:bs, bs:]

    loss = 0.0
    for c in range(num_classes):
        # source hard membership for class c
        s_mask = (source_label == c).float().view(bs, 1)  # (bs,1)
        s_sum = s_mask.sum() + 1e-12
        w_s = s_mask / s_sum  # normalized weights

        # target soft membership for class c
        t_mask = target_prob[:, c].view(bt, 1)  # (bt,1)
        t_sum = t_mask.sum() + 1e-12
        w_t = t_mask / t_sum  # normalized weights

        # class-wise MMD with weights
        term_ss = (w_s @ w_s.t() * K_ss).sum()
        term_tt = (w_t @ w_t.t() * K_tt).sum()
        term_st = (w_s @ w_t.t() * K_st).sum()

        loss = loss + term_ss + term_tt - 2.0 * term_st

    return loss / num_classes

# -----------------------------
# One-batch sanity check
# -----------------------------
model.train()

xs, ys_b = next(iter(src_loader))
xt = next(iter(tgt_tr_loader))

xs = xs.to(device)
ys_b = ys_b.to(device)
xt = xt.to(device)

# forward
fs, logits_s = model(xs)
ft, logits_t = model(xt)

prob_t = F.softmax(logits_t, dim=1)

# losses
ce_loss = F.cross_entropy(logits_s, ys_b)
align_loss = lmmd_loss(fs, ft, ys_b, prob_t, num_classes=4)

print("CE loss   :", float(ce_loss.detach().cpu()))
print("LMMD loss :", float(align_loss.detach().cpu()))


CE loss   : 1.3540349006652832
LMMD loss : 1.3099596500396729


In [ ]:
import os
import numpy as np
import torch
import torch.nn.functional as F

SAVE_DIR = r"F:\MCT PAPER_FAROOQ"
os.makedirs(SAVE_DIR, exist_ok=True)

# -----------------------------
# Training config
# -----------------------------
EPOCHS = 10   # start small on CPU; later you can increase to 40-60
LR = 1e-3
WEIGHT_DECAY = 1e-4
LAMBDA_LMMD = 1.0

# -----------------------------
# Optimizer
# -----------------------------
model = model.to(device)
opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

# -----------------------------
# Helper: evaluate on target test
# -----------------------------
def evaluate_target(model, loader, device):
    model.eval()
    y_true_list = []
    y_pred_list = []
    y_prob_list = []
    feat_list = []

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)

            feat, logits = model(xb)
            prob = F.softmax(logits, dim=1)
            pred = torch.argmax(prob, dim=1)

            y_true_list.append(yb.cpu().numpy())
            y_pred_list.append(pred.cpu().numpy())
            y_prob_list.append(prob.cpu().numpy())
            feat_list.append(feat.cpu().numpy())

    y_true = np.concatenate(y_true_list, axis=0)
    y_pred = np.concatenate(y_pred_list, axis=0)
    y_prob = np.concatenate(y_prob_list, axis=0)
    feat   = np.concatenate(feat_list, axis=0)

    acc = (y_true == y_pred).mean()
    return acc, y_true, y_pred, y_prob, feat

# -----------------------------
# Training loop
# -----------------------------
print("\n==============================")
print("TRAINING START")
print("==============================")

for ep in range(1, EPOCHS + 1):
    model.train()

    total_loss = 0.0
    total_ce = 0.0
    total_lmmd = 0.0
    steps = 0

    src_iter = iter(src_loader)
    tgt_iter = iter(tgt_tr_loader)

    n_steps = min(len(src_loader), len(tgt_tr_loader))

    for _ in range(n_steps):
        xs, ys_b = next(src_iter)
        xt = next(tgt_iter)

        xs = xs.to(device)
        ys_b = ys_b.to(device)
        xt = xt.to(device)

        opt.zero_grad()

        fs, logits_s = model(xs)
        ft, logits_t = model(xt)

        prob_t = F.softmax(logits_t, dim=1)

        ce_loss = F.cross_entropy(logits_s, ys_b)
        align_loss = lmmd_loss(fs, ft, ys_b, prob_t, num_classes=4)

        loss = ce_loss + LAMBDA_LMMD * align_loss
        loss.backward()
        opt.step()

        total_loss += float(loss.detach().cpu())
        total_ce += float(ce_loss.detach().cpu())
        total_lmmd += float(align_loss.detach().cpu())
        steps += 1

    # evaluate per epoch
    acc, _, _, _, _ = evaluate_target(model, tgt_te_loader, device)

    print(f"Epoch [{ep:02d}/{EPOCHS}] "
          f"Loss={total_loss/steps:.4f} | CE={total_ce/steps:.4f} | LMMD={total_lmmd/steps:.4f} "
          f"| TargetTest Acc={acc*100:.2f}%")

print("\n==============================")
print("TRAINING END")
print("==============================")

# -----------------------------
# Final evaluation + save for Step 8 plots
# -----------------------------
final_acc, y_true, y_pred, y_prob, feat = evaluate_target(model, tgt_te_loader, device)

np.save(os.path.join(SAVE_DIR, "y_true.npy"), y_true)
np.save(os.path.join(SAVE_DIR, "y_pred.npy"), y_pred)
np.save(os.path.join(SAVE_DIR, "y_prob.npy"), y_prob)
np.save(os.path.join(SAVE_DIR, "feat.npy"), feat)

print("\nFinal Target Test Accuracy:", f"{final_acc*100:.2f}%")
print("Saved to:", SAVE_DIR)
print("  y_true.npy")
print("  y_pred.npy")
print("  y_prob.npy")
print("  feat.npy")



TRAINING START
Epoch [01/10] Loss=0.2089 | CE=0.1694 | LMMD=0.0395 | TargetTest Acc=100.00%
Epoch [02/10] Loss=0.0041 | CE=0.0009 | LMMD=0.0032 | TargetTest Acc=100.00%
Epoch [03/10] Loss=0.0037 | CE=0.0008 | LMMD=0.0029 | TargetTest Acc=100.00%
Epoch [04/10] Loss=0.0038 | CE=0.0009 | LMMD=0.0028 | TargetTest Acc=100.00%
Epoch [05/10] Loss=0.0029 | CE=0.0009 | LMMD=0.0020 | TargetTest Acc=100.00%
Epoch [06/10] Loss=0.0013 | CE=0.0004 | LMMD=0.0009 | TargetTest Acc=100.00%
Epoch [07/10] Loss=0.0030 | CE=0.0011 | LMMD=0.0019 | TargetTest Acc=100.00%
Epoch [08/10] Loss=0.0014 | CE=0.0004 | LMMD=0.0010 | TargetTest Acc=100.00%
Epoch [09/10] Loss=0.0013 | CE=0.0004 | LMMD=0.0009 | TargetTest Acc=100.00%


In [ ]:
# =========================================================
# STEP 8: Paper-quality Confusion Matrix + ROC + t-SNE
# Loads from: F:\MCT PAPER_FAROOQ\
# Saves to  : F:\MCT PAPER_FAROOQ\figs\
# =========================================================

import os
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc
from sklearn.preprocessing import label_binarize
from sklearn.manifold import TSNE

# -----------------------------
# Paths
# -----------------------------
BASE = r"F:\MCT PAPER_FAROOQ"
FIG_DIR = os.path.join(BASE, "figs")
os.makedirs(FIG_DIR, exist_ok=True)
print("Saving STEP 8 outputs to:", FIG_DIR)

# -----------------------------
# Load final outputs (from Step 7)
# -----------------------------
y_true = np.load(os.path.join(BASE, "y_true.npy")).astype(np.int64)
y_pred = np.load(os.path.join(BASE, "y_pred.npy")).astype(np.int64)
y_prob = np.load(os.path.join(BASE, "y_prob.npy")).astype(np.float32)
feat   = np.load(os.path.join(BASE, "feat.npy")).astype(np.float32)

class_names = ["BF", "GF", "N", "TF"]
n_classes = 4

# -----------------------------
# Confusion Matrix (same style)
# -----------------------------
def save_confusion(cm, name):
    path = os.path.join(FIG_DIR, f"{name}_ConfusionMatrix_2000dpi.png")
    plt.figure(figsize=(5.5, 5.5))
    plt.imshow(cm, cmap="YlGnBu")
    ticks = np.arange(n_classes)
    plt.xticks(ticks, class_names, fontsize=14, fontweight="bold")
    plt.yticks(ticks, class_names, fontsize=14, fontweight="bold")

    plt.gca().set_xticks(np.arange(-.5, n_classes, 1), minor=True)
    plt.gca().set_yticks(np.arange(-.5, n_classes, 1), minor=True)
    plt.grid(which="minor", color="black", linestyle="-", linewidth=1.5)
    plt.tick_params(which="minor", bottom=False, left=False)

    thresh = cm.max() * 0.55
    for i in range(n_classes):
        for j in range(n_classes):
            plt.text(j, i, f"{cm[i, j]}",
                     ha="center", va="center",
                     fontsize=18, fontweight="bold",
                     color="white" if cm[i, j] > thresh else "black")

    plt.xlabel("Predicted Class", fontsize=15, fontweight="bold")
    plt.ylabel("True Class", fontsize=15, fontweight="bold")
    plt.tight_layout()
    plt.savefig(path, dpi=2000, bbox_inches="tight")
    plt.close()
    return path

# -----------------------------
# ROC (same style)
# -----------------------------
def save_roc(y_true, y_prob, name):
    path = os.path.join(FIG_DIR, f"{name}_ROC_2000dpi.png")
    y_true_bin = label_binarize(y_true, classes=np.arange(n_classes))

    plt.figure(figsize=(6.5, 5.5))
    auc_list = []
    for c in range(n_classes):
        fpr, tpr, _ = roc_curve(y_true_bin[:, c], y_prob[:, c])
        roc_auc = auc(fpr, tpr)
        auc_list.append(roc_auc)
        plt.plot(fpr, tpr, linewidth=2.5, label=f"{class_names[c]} (AUC={roc_auc:.3f})")

    plt.plot([0, 1], [0, 1], "k--", linewidth=1.8, label="Chance")
    plt.xlabel("False Positive Rate", fontsize=13, fontweight="bold")
    plt.ylabel("True Positive Rate", fontsize=13, fontweight="bold")
    leg = plt.legend(fontsize=11, frameon=True)
    leg.get_frame().set_edgecolor("black")
    leg.get_frame().set_linewidth(1.5)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(path, dpi=2000, bbox_inches="tight")
    plt.close()
    return path, auc_list

# -----------------------------
# t-SNE (same style)
# -----------------------------
def save_tsne(features, y_true, name):
    path = os.path.join(FIG_DIR, f"{name}_tSNE_2000dpi.png")
    tsne = TSNE(n_components=2, perplexity=25, init="pca", learning_rate="auto", random_state=42)
    Z = tsne.fit_transform(features)

    markers = ["o", "s", "^", "D"]
    plt.figure(figsize=(6.5, 6.5))
    for c, cname in enumerate(class_names):
        idx = (y_true == c)
        plt.scatter(Z[idx, 0], Z[idx, 1],
                    s=95, marker=markers[c],
                    edgecolors="black", linewidths=1.2,
                    alpha=0.9, label=cname)

    plt.xlabel("t-SNE 1", fontsize=13, fontweight="bold")
    plt.ylabel("t-SNE 2", fontsize=13, fontweight="bold")
    leg = plt.legend(fontsize=12, frameon=True)
    leg.get_frame().set_edgecolor("black")
    leg.get_frame().set_linewidth(2.0)
    leg.get_frame().set_alpha(1.0)

    plt.tight_layout()
    plt.savefig(path, dpi=2000, bbox_inches="tight")
    plt.close()
    return path

# -----------------------------
# Generate + Save everything
# -----------------------------
name = "SKN_LMMD_Final"

acc = (y_pred == y_true).mean()
print(f"Target Test Accuracy: {acc*100:.2f}%")

# report
rep = classification_report(y_true, y_pred, target_names=class_names, digits=4)
rep_path = os.path.join(BASE, f"{name}_classification_report.txt")
with open(rep_path, "w", encoding="utf-8") as f:
    f.write(f"{name}\n")
    f.write(f"Accuracy: {acc*100:.2f}%\n\n")
    f.write(rep)

# confusion matrix
cm = confusion_matrix(y_true, y_pred)
cm_path = save_confusion(cm, name)

# ROC
roc_path, auc_list = save_roc(y_true, y_prob, name)
auc_path = os.path.join(BASE, f"{name}_auc_scores.txt")
with open(auc_path, "w", encoding="utf-8") as f:
    for i, cname in enumerate(class_names):
        f.write(f"{cname}: {auc_list[i]:.4f}\n")

# t-SNE
tsne_path = save_tsne(feat, y_true, name)

print("\nSaved:")
print(" ", os.path.basename(cm_path))
print(" ", os.path.basename(roc_path))
print(" ", os.path.basename(tsne_path))
print(" ", os.path.basename(rep_path))
print(" ", os.path.basename(auc_path))
print("\nSTEP 8 complete. All figures are in:", FIG_DIR)
